# Solution B3 · Rare classes: weights, `balanced_subsample` and a threshold chosen with the OOB

**Type**: Implement with scikit-learn

## Objective

On a problem with 3% of positives compare four forests (base, `class_weight="balanced"`, `"balanced_subsample"`, minimum leaf 10) with average precision and F1; choose the **threshold** from the `oob_decision_function_` (without cross-validation and without touching the test set) and check that it improves the F1 over 0.5;
measure the effect on the mean probabilities.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(12000, n_features=12, n_informative=5, n_redundant=0, weights=[0.97], class_sep=0.9, flip_y=0.01, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)
configs = {"base": {}, "balanced": {"class_weight": "balanced"}, "balanced_subsample": {"class_weight": "balanced_subsample"}, "leaf10": {"min_samples_leaf": 10}}
print("positives: training", int(y_tr.sum()), "| test", int(y_te.sum()))

## Commented solution

In [ ]:
def f1_threshold(y, p):
    ts = np.linspace(0.02, 0.9, 89)
    return float(ts[int(np.argmax([f1_score(y, p >= t) for t in ts]))])

In [ ]:
rows = {}
for name, kw in configs.items():
    f = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1, **kw).fit(X_tr, y_tr)
    p = f.predict_proba(X_te)[:, 1]
    threshold = f1_threshold(y_tr, f.oob_decision_function_[:, 1])                  # threshold chosen on the out-of-bag probabilities of the TRAINING set
    rows[name] = {"ap": average_precision_score(y_te, p), "auc": roc_auc_score(y_te, p), "f1_05": f1_score(y_te, p >= 0.5), "mean_p": p.mean(),
                  "oob_threshold": threshold, "oob_f1": f1_score(y_te, p >= threshold)}
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
best_f1 = table["oob_f1"].idxmax()
threshold_gain = table.loc["base", "oob_f1"] - table.loc["base", "f1_05"]
threshold_far_from_05 = bool(table.loc["base", "oob_threshold"] < 0.4)
print(f"best F1 with the OOB threshold: {best_f1} | gain of the threshold for the base forest: {threshold_gain:+.3f} (threshold {table.loc['base', 'oob_threshold']:.2f})")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["ap", "auc", "f1_05", "mean_p", "oob_threshold", "oob_f1"] and list(table.index) == list(configs), "table with six columns and four configurations", "ap, auc, f1_05, mean_p, oob_threshold, oob_f1")
    check(table["auc"].min() > 0.83 and table["ap"].min() > 0.5 and table["ap"].max() - table["ap"].min() < 0.06, "AUC over 0.83 and average precision 0.5-0.65 for all: the configurations rank almost the same way", "the forest already handles the imbalance well")
    check((table["oob_f1"] > table["f1_05"]).all(), "with the OOB threshold the F1 on the test set improves over threshold 0.5 for all the configurations", "oob_decision_function_[:, 1]")
    check(table.loc["base", "oob_threshold"] < 0.4 and threshold_far_from_05 is True and threshold_gain > 0.04, "the OOB threshold for the base forest is below 0.4 and gains over 0.04 of F1", "oob_f1 - f1_05 for the base configuration")
    check(best_f1 in configs and table.loc[best_f1, "oob_f1"] == table["oob_f1"].max(), "configuration with the highest F1", "idxmax of oob_f1")
    check(table.loc["base", "f1_05"] > table.loc["balanced", "f1_05"], "at threshold 0.5 the 'balanced' weights worsen the F1 (the base model has a much higher precision)", "compare f1_05")
done()

## What you learned

- **The base forest is already robust to imbalance:** AUC 0.86 and average precision 0.59, and the class weights change little (AP 0.60 with `balanced`). But **at threshold 0.5** a rare positive rarely receives a fraction of votes above 0.5: recall of a third and F1 0.50.
- **A threshold from the OOB improves the F1 without touching the test set:** 0.29 on the out-of-bag probabilities (out-of-sample predictions of the whole training set) takes the F1 from 0.50 to 0.58 on the test set. It is a direct application of the idea of M4 (choose the threshold on the training set with out-of-sample probabilities), but without `cross_val_predict`: the OOB is already there.
- **The class weights (`balanced`) do not help** with the optimized threshold and worsen the F1 at 0.5 (more false positives): with a forest it is almost always better to leave the probabilities free and tune the threshold. It is the same result as M4 F2 with another model.
- With a small number of positives (204) the estimate of the threshold is noisy: here it is fine because the OOB has a large number of examples, but with few positives a coarser grid of thresholds is better and do not "optimize" to the third decimal.

## Common pitfall

Rebalancing the classes (`class_weight`, oversampling) as a first move with a forest. The forest already ranks the cases well (high AUC): the problem is where to cut, that is the threshold, and that is solved without distorting the probabilities.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you choose the threshold of a random forest on an imbalanced problem?</summary>

I use the out-of-bag probabilities of the training set (oob_decision_function_), which are out-of-sample predictions: I choose the threshold on them based on costs or F1 and then validate it once on the test set. I avoid rebalancing at the start: the forest already ranks well, and the weights distort the probabilities. If I need calibrated
probabilities I recalibrate them (sigmoid or isotonic) with data not used for training.

</details>